# Traffic Demand Prediction
This notebook builds predictive models (XGBoost, LightGBM, CatBoost) to forecast traffic demand based on spatiotemporal and weather features. 

We will:
1. **Preprocess the data**: Extract hour/minute from `timestamp`, impute missing values, and type cast categorical features.
2. **Hyperparameter Tuning**: Use `Optuna` to find the best parameters for each model.
3. **Cross-Validation**: Use K-Fold cross validation to reliably estimate model performance during Optuna tuning.
4. **Ensembling**: Blend the predictions of all three models to create a robust final submission.


In [28]:
!pip install optuna xgboost lightgbm catboost scikit-learn pandas numpy

## 1. Setup and Data Loading
In this section, we import the necessary libraries and load the `train.csv` and `test.csv` datasets. We ignore warnings to keep the output clean.

In [29]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# Load Data
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')


## 2. Preprocessing & Feature Engineering
Here, we define our preprocessing logic:
- The `timestamp` column (e.g., `0:0`) is split into numerical `hour` and `minute` features. This allows tree-based models to split effectively on time.
- Missing numerical values (`Temperature`) are filled using the median from the training set.
- Missing categorical values are filled with the string `'Unknown'` and explicitly converted to Pandas `category` types, which LightGBM and XGBoost can handle natively.


In [30]:
def preprocess_data(df, is_train=True):
    df_copy = df.copy()
    
    # 1. Spatiotemporal Features
    # timestamp is like "0:0", "0:15", etc.
    df_copy['hour'] = df_copy['timestamp'].apply(lambda x: int(x.split(':')[0]) if pd.notnull(x) else -1)
    df_copy['minute'] = df_copy['timestamp'].apply(lambda x: int(x.split(':')[1]) if pd.notnull(x) else -1)
    
    # Drop original timestamp
    df_copy.drop(['timestamp'], axis=1, inplace=True)
    
    # 2. Missing Value Imputation
    df_copy['Temperature'] = df_copy['Temperature'].fillna(df_copy['Temperature'].median() if is_train else 25.0) 
    
    cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
    for col in cat_cols:
        df_copy[col] = df_copy[col].fillna('Unknown')
        df_copy[col] = df_copy[col].astype('category')
        
    if 'NumberofLanes' in df_copy.columns:
        df_copy['NumberofLanes'] = df_copy['NumberofLanes'].fillna(-1)
        
    return df_copy

# Apply Preprocessing
train_temp_median = train['Temperature'].median()
train_clean = preprocess_data(train, is_train=True)
test_clean = preprocess_data(test, is_train=False)
# Use train median to fill test temperature missing values to prevent data leakage
test_clean['Temperature'] = test_clean['Temperature'].fillna(train_temp_median)

X = train_clean.drop(['Index', 'demand'], axis=1)
y = train_clean['demand']
X_test = test_clean.drop(['Index'], axis=1)

cat_features = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']


## 3. Hyperparameter Tuning with Optuna
Optuna is an automatic hyperparameter optimization software. 
Below is a setup for tuning LightGBM. It uses K-Fold Cross Validation (5 folds) to evaluate each set of parameters against our specific evaluation metric (`max(0, 100 * r2_score)`).

In [31]:
# Custom evaluation metric function based on hackathon rules
def eval_metric(y_true, y_pred):
    return max(0, 100 * r2_score(y_true, y_pred))

# Example Optuna Objective for LightGBM
def lgb_objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': 300
    }
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        model = LGBMRegressor(**params, random_state=42)
        model.fit(X_tr, y_tr, categorical_feature=cat_features)
        preds = model.predict(X_va)
        score = eval_metric(y_va, preds)
        scores.append(score)
    return np.mean(scores)

# To run tuning:
# study_lgb = optuna.create_study(direction='maximize')
# study_lgb.optimize(lgb_objective, n_trials=10)
# best_lgb_params = study_lgb.best_params


## 4. Model Training
We define our three models (LightGBM, CatBoost, XGBoost) using parameters that are already known to work reasonably well. We train all three models on the full training dataset and generate predictions for the `test.csv` dataset.

In [32]:
# Using pre-tuned parameters for demonstration
# You can replace these with `study.best_params` if you run Optuna tuning
lgb_params = {'learning_rate': 0.05, 'num_leaves': 60, 'max_depth': 8, 'n_estimators': 500, 'random_state': 42}
cat_params = {'learning_rate': 0.05, 'depth': 6, 'iterations': 500, 'random_seed': 42, 'verbose': False, 'cat_features': cat_features}
xgb_params = {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 500, 'random_state': 42, 'enable_categorical': True, 'tree_method': 'hist'}

# LightGBM
print("Training LightGBM...")
model_lgb = LGBMRegressor(**lgb_params)
model_lgb.fit(X, y, categorical_feature=cat_features)
preds_lgb = model_lgb.predict(X_test)
train_preds_lgb = model_lgb.predict(X)

# CatBoost
print("Training CatBoost...")
model_cat = CatBoostRegressor(**cat_params)
model_cat.fit(X, y)
preds_cat = model_cat.predict(X_test)
train_preds_cat = model_cat.predict(X)

# XGBoost
print("Training XGBoost...")
model_xgb = XGBRegressor(**xgb_params)
model_xgb.fit(X, y)
preds_xgb = model_xgb.predict(X_test)
train_preds_xgb = model_xgb.predict(X)


Training LightGBM...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003820 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1374
[LightGBM] [Info] Number of data points in the train set: 77299, number of used features: 10
[LightGBM] [Info] Start training from score 0.093942
Training CatBoost...
Training XGBoost...


## 5. Training Evaluation
We calculate the R2 score of our final ensemble on the **training data**. 
*(Note: Since the models were trained on this data, this score will be artificially high. It serves primarily as a sanity check to ensure the models learned the training patterns correctly).*

In [33]:
# Blend predictions on the training set
train_preds_blend = (train_preds_lgb + train_preds_cat + train_preds_xgb) / 3.0

# Calculate Hackathon Metric
train_r2_score = eval_metric(y, train_preds_blend)
print(f"Blended Ensemble Training R2 Score: {train_r2_score:.4f}")


Blended Ensemble Training R2 Score: 96.8278


## 6. Ensembling & Submission
We blend the predictions from all three models for the test set using a simple average. 
Finally, we construct a new DataFrame linking the `Index` from `test.csv` to our blended predictions and save it to `submission_blend.csv`.

In [34]:
# Ensembling test predictions: Simple Average Blend
print("Blending Test Predictions...")
preds_blend = (preds_lgb + preds_cat + preds_xgb) / 3.0

# Create Submission
# We create a new dataframe dynamically to avoid shape mismatch issues 
# (sample_submission.csv only has 5 rows, test.csv has 41778 rows)
sub = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds_blend
})

sub.to_csv('submission_blend.csv', index=False)
print("Saved submission_blend.csv")
sub.head()


Blending Test Predictions...
Saved submission_blend.csv


,Index,demand
0,0,0.048985
1,1,0.020329
2,2,0.023532
3,3,0.024219
4,4,0.059973
